In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime
import re
from bs4 import BeautifulSoup
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi
import glob, os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import (
    NoSuchElementException,
    ElementClickInterceptedException,
    StaleElementReferenceException,
)
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support import expected_conditions as EC
from pathlib import Path

In [ ]:
# -----------------------------------------------------
# API 키 설정
# -----------------------------------------------------

# 네이버 블로그
CLIENT_ID = '' # ID 입력
CLIENT_SECRET = '' # P/W

# 유튜브 댓글
api_key = "비밀키"  # 발급받은 API 키 입력
youtube = build("youtube", "v3", developerKey=api_key)

# 카카오 맵
KAKAO_MAP_URL = "https://map.kakao.com/"
SEARCH_QUERY = "100대 명산"

# 네이버 지도
SHARED_LIST_URL = "https://naver.me/GEicSU50"

# 함수

In [ ]:
def clean_html(text):
    """HTML 태그 및 특수문자 제거"""
    if not text:
        return ""
    text = re.sub(r'<[^>]+>', '', text)
    text = text.replace('&nbsp;', ' ').replace('&amp;', '&')
    return re.sub(r'\s+', ' ', text).strip()

def format_date(date_str):
    """YYYYMMDD -> YYYY-MM-DD"""
    if date_str and len(date_str) == 8:
        return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"
    return date_str

def extract_blog_content(url):
    """네이버 블로그 본문 추출"""
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # iframe 확인
        iframe = soup.find('iframe', {'id': 'mainFrame'})
        if iframe and iframe.get('src'):
            iframe_url = iframe['src']
            if not iframe_url.startswith('http'):
                iframe_url = 'https://blog.naver.com' + iframe_url
            time.sleep(0.5)
            response = requests.get(iframe_url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
        
        # 본문 추출
        content = soup.find('div', class_='se-main-container')
        if not content:
            content = soup.find('div', id='postViewArea')
        
        if content:
            for tag in content.find_all(['script', 'style']):
                tag.decompose()
            text = content.get_text(separator='\n', strip=True)
            return re.sub(r'\n{3,}', '\n\n', text)
        return None
    except:
        return None

# -----------------------------------------------------
# 유튜브 댓글 수집 함수
# -----------------------------------------------------
def get_youtube_comments(video_id, max_comments=100):
    comments = []
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )
    while request and len(comments) < max_comments:
        response = request.execute()
        for item in response.get("items", []):
            comment = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
            comments.append(comment)
            if len(comments) >= max_comments:
                break
        request = youtube.commentThreads().list_next(request, response)
    return comments

# -----------------------------------------------------
# 카카오맵
# -----------------------------------------------------
def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    # 필요하면 headless 모드
    # options.add_argument("--headless=new")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )
    return driver


def safe_filename(name: str) -> str:
    """파일명에 쓸 수 없는 문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "_", name)


def generate_unique_filename(base_name: str) -> str:
    """
    base_name: 'OO_지리산_review.csv'
    이미 존재하면 'OO_지리산2_review.csv', 'OO_지리산3_review.csv' ... 생성
    """
    if not os.path.exists(base_name):
        return base_name

    name, ext = os.path.splitext(base_name)  
    counter = 2
    new_name = f"{name}{counter}{ext}"

    while os.path.exists(new_name):
        counter += 1
        new_name = f"{name}{counter}{ext}"

    return new_name

# 1. "100대 명산" 검색 + '장소' 탭으로 이동

def search_100_mountains(driver):
    wait = WebDriverWait(driver, 10)

    driver.get(KAKAO_MAP_URL)
    
    # 검색어 입력
    wait.until(EC.presence_of_element_located((By.ID, "search.keyword.query")))

    search_input = driver.find_element(By.ID, "search.keyword.query")
    search_input.clear()
    search_input.send_keys(SEARCH_QUERY)

    # dimmed 레이어 (팝업창 같은 거) 있으면 숨기기
    try:
        dim = driver.find_element(By.ID, "dimmedLayer")
        driver.execute_script("arguments[0].style.display='none';", dim)
    except NoSuchElementException:
        pass

    # 엔터로 검색 실행
    search_input.send_keys(Keys.ENTER)

    # '장소' 탭 클릭
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#info\\.main\\.options > li")))

    try:
        tabs = driver.find_elements(By.CSS_SELECTOR, "#info\\.main\\.options > li")
        for t in tabs:
            if "장소" in t.text:
                t.click()
                wait.until(EC.presence_of_element_located((By.ID, "info.search.place.list")))
                break
    except Exception as e:
        print("장소 탭 클릭 실패(무시하고 진행):", e)

# -----------------------------------------------------
# 2. 검색 결과 페이지에서 place URL 모두 수집
# -----------------------------------------------------
def collect_all_place_urls(driver):
    place_urls = []
    visited = set()

    while True:  # next(화살표) 묶음 단위로 반복
    
        # 현재 세트에서 페이지 번호 1~5 순회
        for page_no in range(1, 6):
            # 페이지 번호 버튼 찾기
            try:
                page_btn = driver.find_element(
                    By.ID, f"info.search.page.no{page_no}"
                )
            except NoSuchElementException:
                print(f"[페이지 버튼 {page_no}] 없음 → 이 세트 종료")
                break

            # 페이지 이동
            try:
                driver.execute_script("arguments[0].click();", page_btn)
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", page_btn)
            time.sleep(2)

            # 이 페이지의 장소 리스트 수집
            items = driver.find_elements(
                By.CSS_SELECTOR,
                'ul[id="info.search.place.list"] > li.PlaceItem',
            )
            print(f"[현재 세트 페이지 {page_no}] PlaceItem 개수: {len(items)}")

            for li in items:
                # 상세보기 링크 (moreview)에서 place URL 추출
                try:
                    moreview = li.find_element(
                        By.CSS_SELECTOR,
                        'a.moreview[data-id="moreview"]',
                    )
                    href = moreview.get_attribute("href")
                    if href and "place.map.kakao.com" in href:
                        if href not in visited:
                            visited.add(href)
                            place_urls.append(href)
                except NoSuchElementException:
                    continue

        # 세트가 끝나면 next(화살표) 버튼 확인
        try:
            next_btn = driver.find_element(By.ID, "info.search.page.next")
            cls = next_btn.get_attribute("class") or ""
            if "disabled" in cls:
                print("◆ next 버튼 disabled → 모든 페이지 탐색 종료")
                break
            print(">>> next 버튼 클릭 (다음 세트로 이동)")
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(2)
        except NoSuchElementException:
            print("◆ next 버튼 없음 → 탐색 종료")
            break

    print("총 수집된 place URL:", len(place_urls))
    return place_urls

# -----------------------------------------------------
# 3. place 상세 페이지: 후기 탭 + 리뷰 로딩
# -----------------------------------------------------
def go_to_place_and_review_tab(driver, place_url: str):
    """
    place_url 에 접속해서 후기 탭으로 이동하고, km_name(장소명)만 반환
    """
    # 바로 리뷰 탭으로 이동 시도
    if "#review" not in place_url:
        url = place_url + "#review"
    else:
        url = place_url

    driver.get(url)
    time.sleep(3)

    # 장소명
    km_name = None
    try:
        name_elem = driver.find_element(By.CSS_SELECTOR, "h3.tit_place")
        raw_name = name_elem.text.strip()
        km_name = raw_name.replace("장소명", "").strip()
    except NoSuchElementException:
        pass

    # 후기 탭 클릭 (이미 review 해시라도 한 번 더 눌러줌)
    try:
        review_tab = driver.find_element(By.CSS_SELECTOR, 'a.link_tab[href="#review"]')
        review_tab.click()
        time.sleep(2)
    except Exception:
        pass

    return km_name


def load_all_reviews(driver, max_round: int = 80, sleep_sec: float = 1.0):
    """
    window 전체를 스크롤하면서 ul.list_review > li 개수가 늘지 않을 때까지 반복
    """
    prev_count = -1
    same_count_rounds = 0

    for r in range(max_round):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(sleep_sec)

        try:
            count = driver.execute_script(
                "return document.querySelectorAll('ul.list_review > li').length;"
            )
        except Exception:
            count = 0

        print(f"  [리뷰 스크롤 {r+1}] 현재 리뷰 개수: {count}")

        if count == prev_count:
            same_count_rounds += 1
            if same_count_rounds >= 3:
                print("  → 더 이상 리뷰가 늘어나지 않아 스크롤 종료")
                break
        else:
            same_count_rounds = 0
            prev_count = count


def click_all_more_buttons(driver, max_round: int = 5, sleep_sec: float = 0.2):
    """
    리뷰 내 '더보기' 버튼 모두 클릭
    - span.btn_more : 긴 리뷰 본문 펼치기
    - a.link_more   : (혹시 있을 수 있는) 링크 타입 더보기
    """
    selectors = [
        "span.btn_more",
        "a.link_more",
    ]

    for round_i in range(max_round):
        clicked_any = False

        for sel in selectors:
            try:
                container = driver.find_element(
                    By.CSS_SELECTOR,
                    "div.section_comm.section_review div.group_review",
                )
            except NoSuchElementException:
                container = driver

            more_buttons = container.find_elements(By.CSS_SELECTOR, sel)
            print(f"  [더보기 라운드 {round_i+1}] selector={sel} 버튼 개수: {len(more_buttons)}")

            for btn in more_buttons:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(sleep_sec)
                        clicked_any = True
                except (StaleElementReferenceException, ElementClickInterceptedException):
                    continue

        if not clicked_any:
            break

# -----------------------------------------------------
# 4. place 페이지 HTML → 리뷰 DataFrame 생성
# -----------------------------------------------------
def parse_reviews_from_page_source(page_source: str, km_name: str) -> pd.DataFrame:
    soup = BeautifulSoup(page_source, "html.parser")

    items = soup.select("ul.list_review > li")
    print(f"  ▶ 파싱된 리뷰 개수: {len(items)}")

    rows = []

    for item in items:
        # 작성자
        author_tag = item.select_one(".name_user")
        if author_tag:
            author_text = author_tag.get_text(strip=True)
            author_text = author_text.replace("리뷰어 이름, ", "")
        else:
            author_text = None

        # 별점
        stars = item.select(".wrap_grade .figure_star.on")
        rating_val = len(stars) if stars else None

        # 날짜
        date_tag = item.select_one(".txt_date")
        date_text = date_tag.get_text(strip=True) if date_tag else None

        # 본문 (전체 리뷰 우선, 없으면 요약본)
        content_tag = item.select_one("p.desc_review")
        if content_tag is None:
            content_tag = item.select_one("div.wrap_review p")
        content_text = content_tag.get_text(strip=True) if content_tag else None

        rows.append(
            {
                "km_name": km_name,
                "km_author": author_text,
                "km_rating": rating_val,
                "km_date": date_text,
                "km_content": content_text,
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df.insert(0, "km_index", range(1, len(df) + 1))
    return df

# -----------------------------------------------------
# 네이버지도
# 0. 드라이버 / 유틸
# -----------------------------------------------------

def korean_date_to_iso(text: str):
    """
    '2025년 8월 2일 토요일' -> '2025-08-02'
    """
    m = re.search(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일", text)
    if not m:
        return None
    y, mth, d = m.groups()
    return f"{int(y):04d}-{int(mth):02d}-{int(d):02d}"

# -----------------------------------------------------
# 1. 리스트 iframe / 상세 iframe 찾기
# -----------------------------------------------------
def find_list_frame_index(driver):
    """
    100대 명산 카드(div._card_area_yih1d_10)가 있는 iframe index 찾기.
    없으면 None (top-level).
    """
    time.sleep(2)
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    print(f"▶ 발견한 iframe 개수: {len(iframes)}")

    if not iframes:
        cards = driver.find_elements(By.CSS_SELECTOR, "div._card_area_yih1d_10")
        print(f"  - top-level 카드 개수: {len(cards)}")
        if cards:
            print("✅ 카드가 top-level 에 있음 (frame_index=None)")
            return None

    for idx, frame in enumerate(iframes):
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame(frame)
            cards = driver.find_elements(By.CSS_SELECTOR, "div._card_area_yih1d_10")
            print(f"  - iframe[{idx}] 카드 개수: {len(cards)}")
            if cards:
                driver.switch_to.default_content()
                print(f"✅ 100대 명산 카드가 있는 iframe index: {idx}")
                return idx
        except Exception:
            driver.switch_to.default_content()
            continue

    driver.switch_to.default_content()
    print("❌ 카드가 들어있는 iframe을 찾지 못했어요.")
    return None


def switch_to_list_frame(driver, frame_index):
    driver.switch_to.default_content()
    if frame_index is None:
        return
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    if frame_index < len(iframes):
        driver.switch_to.frame(iframes[frame_index])


def find_detail_frame_index(driver):
    """
    산 카드 클릭 후, 리뷰 탭/리뷰 내용이 들어있는 상세 패널 iframe index 찾기.
    """
    time.sleep(2)
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    print(f"▶ 상세 페이지 iframe 개수: {len(iframes)}")

    # top-level 먼저 체크
    driver.switch_to.default_content()
    try:
        tabs = driver.find_elements(By.XPATH, "//span[text()='리뷰']/ancestor::a")
        nicks = driver.find_elements(By.CSS_SELECTOR, "span.pui__NMi-Dp")
        if tabs or nicks:
            print("✅ 상세 패널이 top-level 에 있음 (detail_frame_index=None)")
            return None
    except Exception:
        pass

    for idx, frame in enumerate(iframes):
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame(frame)
            tabs = driver.find_elements(By.XPATH, "//span[text()='리뷰']/ancestor::a")
            nicks = driver.find_elements(By.CSS_SELECTOR, "span.pui__NMi-Dp")
            if tabs or nicks:
                driver.switch_to.default_content()
                print(f"✅ 상세 패널이 있는 iframe index: {idx}")
                return idx
        except Exception:
            driver.switch_to.default_content()
            continue

    driver.switch_to.default_content()
    print("❌ 상세 패널 iframe을 찾지 못했어요.")
    return None


def switch_to_detail_frame(driver, frame_index):
    '''
    지금 페이지에서 iframe 구조를 리셋한 뒤, 원하는 순서의 iframe 안으로 들어가는 함수
    '''
    driver.switch_to.default_content()
    if frame_index is None:
        return
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    if frame_index < len(iframes):
        driver.switch_to.frame(iframes[frame_index])

# -----------------------------------------------------
# 2. 리스트 스크롤 → 100개 카드 정보 (순번 + 이름) 수집
# -----------------------------------------------------
def collect_all_mountain_infos(driver, list_frame_index, max_round=50, sleep_sec=1.0):
    """
    스크롤 끝까지 내린 뒤,
    최종 카드 목록을 한 번 더 읽어서
    [{'order': 0, 'name': '가리산'}, {'order': 1, 'name': '지리산'}, ...] 반환.
    """
    prev_count = -1
    same_rounds = 0

    for r in range(max_round):
        switch_to_list_frame(driver, list_frame_index)
        cards = driver.find_elements(By.CSS_SELECTOR, "div._card_area_yih1d_10")
        count = len(cards)
        print(f"[리스트 스크롤 {r+1}] 현재 카드 개수: {count}")

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(sleep_sec)

        if count == prev_count:
            same_rounds += 1
            if same_rounds >= 3:
                print("  → 더 이상 카드가 늘어나지 않아 스크롤 종료")
                break
        else:
            same_rounds = 0
            prev_count = count

    # 스크롤 끝난 후 최종 카드 목록 한 번 더 읽기
    switch_to_list_frame(driver, list_frame_index)
    cards = driver.find_elements(By.CSS_SELECTOR, "div._card_area_yih1d_10")

    mountains = []
    for i, c in enumerate(cards):
        try:
            title_el = c.find_element(By.CSS_SELECTOR, "strong._main_title_yih1d_68")
            nm_name = title_el.text.strip()
        except NoSuchElementException:
            nm_name = None
        mountains.append({"order": i, "name": nm_name})

    driver.switch_to.default_content()
    print(f"총 수집된 카드 개수: {len(mountains)}")
    print("예시:", mountains[:5])
    return mountains

# -----------------------------------------------------
# 3. 리뷰 탭 / 스크롤 / 더보기
# -----------------------------------------------------
def open_review_tab(driver, detail_frame_index):
    switch_to_detail_frame(driver, detail_frame_index)
    wait = WebDriverWait(driver, 15)
    try:
        review_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//span[text()='리뷰']/ancestor::a")
            )
        )
        review_tab.click()
        time.sleep(2)
    except Exception as e:
        print("  ▶ 리뷰 탭 클릭 실패:", e)
    finally:
        driver.switch_to.default_content()


def scroll_reviews_and_load_more(driver, detail_frame_index, max_round=40, sleep_sec=1.0):
    prev_count = -1
    same_rounds = 0

    for r in range(max_round):
        switch_to_detail_frame(driver, detail_frame_index)

        # '펼쳐서 더보기' 버튼 있으면 클릭
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, "a.fvwqf")
            if more_btn.is_displayed():
                print(f"  [리뷰 스크롤 {r+1}] '펼쳐서 더보기' 클릭")
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(sleep_sec)
        except NoSuchElementException:
            pass

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(sleep_sec)

        try:
            count = driver.execute_script(
                "return document.querySelectorAll('span.pui__NMi-Dp').length;"
            )
        except Exception:
            count = 0

        print(f"  [리뷰 스크롤 {r+1}] 현재 리뷰(닉네임) 개수: {count}")

        if count == prev_count:
            same_rounds += 1
            if same_rounds >= 3:
                print("  → 리뷰 개수 증가 멈춤 → 스크롤 종료")
                break
        else:
            same_rounds = 0
            prev_count = count

    driver.switch_to.default_content()


def click_all_review_more_buttons(driver, detail_frame_index, max_round=5, sleep_sec=0.3):
    """
    각 리뷰 카드 안 '더보기' 버튼(본문 펼치기)을 모두 클릭
    - a.pui__wFzIYl[data-pui-click-code='rvshowmore']
    """
    for r in range(max_round):
        switch_to_detail_frame(driver, detail_frame_index)
        try:
            more_btns = driver.find_elements(
                By.CSS_SELECTOR,
                "a.pui__wFzIYl[data-pui-click-code='rvshowmore']",
            )
        except Exception:
            more_btns = []

        print(f"  [본문 더보기 라운드 {r+1}] 버튼 개수: {len(more_btns)}")

        if not more_btns:
            break

        clicked_any = False
        for btn in more_btns:
            try:
                if btn.is_displayed():
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(sleep_sec)
                    clicked_any = True
            except (StaleElementReferenceException, ElementClickInterceptedException):
                continue

        if not clicked_any:
            break

    driver.switch_to.default_content()

# -----------------------------------------------------
# 4. HTML 파싱 (리뷰 카드 단위로 작성자/날짜/본문 묶기)
# -----------------------------------------------------
def parse_naver_reviews(page_source, nm_name):
    soup = BeautifulSoup(page_source, "html.parser")

    # 리뷰닉네임 span 기준으로 카드 단위로 묶기
    nick_tags = soup.select("span.pui__NMi-Dp")
    print(f"  ▶ 닉네임 태그 개수: {len(nick_tags)}")

    rows = []
    seen = set()  # (author, date, content) 중복 제거용

    for nick in nick_tags:
        author = nick.get_text(strip=True)
        if not author:
            continue

        # 이 닉네임이 속한 리뷰 카드(div)를 위로 올라가며 찾기
        card = nick
        while card and not card.select_one("div.pui__vn15t2"):
            card = card.parent
        if not card:
            continue

        # 날짜 span.pui__blind (YYYY년 M월 D일…) 찾기
        date_tag = card.find(
            "span",
            class_="pui__blind",
            string=re.compile(r"\d{4}년\s*\d{1,2}월\s*\d{1,2}일"),
        )
        if not date_tag:
            continue
        nm_date = korean_date_to_iso(date_tag.get_text(strip=True))

        # 본문 div.pui__vn15t2 안의 텍스트 (더보기 클릭 후 전체가 들어있다고 가정)
        content_div = card.select_one("div.pui__vn15t2")
        if not content_div:
            continue

        a_tags = content_div.select("a[data-pui-click-code='rvshowmore']")
        main_a = None

        # '더보기' 버튼(a.pui__wFzIYl)은 제외하고 본문 a 선택
        for a in a_tags:
            classes = a.get("class") or []
            if "pui__wFzIYl" not in classes:
                main_a = a
                break

        if main_a is None:
            if a_tags:
                main_a = a_tags[0]
            else:
                main_a = content_div

        nm_content = main_a.get_text(" ", strip=True)
        if not nm_content:
            continue

        key = (author, nm_date, nm_content)
        if key in seen:
            continue
        seen.add(key)

        rows.append(
            {
                "nm_name": nm_name,
                "nm_author": author,
                "nm_date": nm_date,
                "nm_content": nm_content,
            }
        )

    print(f"  ▶ 파싱된 리뷰 개수(중복 제거 후): {len(rows)}")

    df = pd.DataFrame(rows)
    if not df.empty:
        df.insert(0, "nm_index", range(1, len(df) + 1))
    return df

def main_km():
    driver = create_driver()

    try:
        # 1) 검색 + 장소 탭
        search_100_mountains(driver)

        # 2) 모든 place URL 수집
        place_urls = collect_all_place_urls(driver)
        print("수집된 place URL 예시:", place_urls[:5])

        main_handle = driver.current_window_handle

        # 3) 각 장소별 리뷰 크롤링
        for idx, url in enumerate(place_urls, start=1):
            print(f"\n===== [{idx}/{len(place_urls)}] {url} =====")

            # 새 탭 열고 place 이동 + 후기 탭
            driver.switch_to.new_window("tab")
            km_name = go_to_place_and_review_tab(driver, url)
            print("  km_name:", km_name)

            # 리뷰 리스트 존재 여부 대략 체크
            try:
                driver.execute_script(
                    "return document.querySelectorAll('ul.list_review > li').length;"
                )
            except Exception:
                print("  ▶ 리뷰 리스트를 찾지 못해 스킵")
                driver.close()
                driver.switch_to.window(main_handle)
                continue

            # 리뷰 전체 로딩 + 더보기 클릭
            load_all_reviews(driver, max_round=80, sleep_sec=1.0)
            click_all_more_buttons(driver, max_round=5, sleep_sec=0.2)

            # HTML 파싱
            page_source = driver.page_source
            df_reviews = parse_reviews_from_page_source(page_source, km_name)

            # CSV 저장 (산별 1파일, 이름 중복 시 숫자 붙이기)
            if not df_reviews.empty and km_name:
                safe_name = safe_filename(km_name)
                base_fname = f"km_{safe_name}_review.csv"
                fname = generate_unique_filename(base_fname)

                df_reviews.to_csv(fname, index=False, encoding="utf-8-sig")
                print(f"  ▶ CSV 저장 완료: {fname} (rows={len(df_reviews)})")
            else:
                print("  ▶ 리뷰 없음 또는 이름 없음 → 파일 미생성")

            # 탭 닫고 검색 결과 페이지로 복귀
            driver.close()
            driver.switch_to.window(main_handle)
            time.sleep(1)

    finally:
        driver.quit()

def main_nm():
    driver = create_driver()
    wait = WebDriverWait(driver, 20)

    try:
        # 공유 리스트 페이지 열기
        driver.get(SHARED_LIST_URL)
        time.sleep(5)

        # 리스트 iframe index
        list_frame_index = find_list_frame_index(driver)

        # 카드 100개 정보(순번 + 이름) 수집
        mountain_infos = collect_all_mountain_infos(driver, list_frame_index)

        for idx, info in enumerate(mountain_infos, start=1):
            nm_name = info["name"]
            card_order = info["order"]
            print(f"\n===== [{idx}/{len(mountain_infos)}] {nm_name} (order={card_order}) =====")

            # 뒤로가기 해서 항상 리스트 화면 기준으로
            if idx > 1:
                driver.back()
                time.sleep(3)

            # 리스트 프레임 진입 후, 순번 기반으로 카드 클릭
            switch_to_list_frame(driver, list_frame_index)
            card_xpath = f"(//div[contains(@class,'_card_area')])[{card_order + 1}]"

            try:
                card_el = wait.until(
                    EC.element_to_be_clickable((By.XPATH, card_xpath))
                )
                driver.execute_script("arguments[0].click();", card_el)
                print(f"  ▶ '{nm_name}' 카드 클릭 (order={card_order})")
                time.sleep(3)
            except Exception as e:
                print(f"  ▶ '{nm_name}' (order={card_order}) 카드 클릭 실패, 스킵: {e}")
                driver.switch_to.default_content()
                continue

            driver.switch_to.default_content()

            # 상세 패널 iframe index
            detail_frame_index = find_detail_frame_index(driver)

            # 리뷰 탭 열기
            open_review_tab(driver, detail_frame_index)

            # 리뷰 존재 여부 대략 체크
            switch_to_detail_frame(driver, detail_frame_index)
            try:
                nick_count = driver.execute_script(
                    "return document.querySelectorAll('span.pui__NMi-Dp').length;"
                )
            except Exception:
                nick_count = 0
            driver.switch_to.default_content()

            if nick_count == 0:
                print("  ▶ 리뷰가 없는 장소로 판단, 스킵")
                continue

            # 리뷰 전체 로딩 (스크롤 + '펼쳐서 더보기')
            scroll_reviews_and_load_more(driver, detail_frame_index, max_round=40, sleep_sec=1.0)

            # 각 리뷰 카드 내 '더보기' 버튼 클릭 (본문 다 펼치기)
            click_all_review_more_buttons(driver, detail_frame_index, max_round=5, sleep_sec=0.3)

            # HTML 파싱
            switch_to_detail_frame(driver, detail_frame_index)
            page_source = driver.page_source
            driver.switch_to.default_content()

            df_reviews = parse_naver_reviews(page_source, nm_name)

            # CSV 저장 (동명이산은 파일명 뒤에 숫자 붙여서 모두 저장)
            if not df_reviews.empty:
                safe_name = safe_filename(nm_name)
                base_fname = f"nm_{safe_name}_review.csv"
                fname = generate_unique_filename(base_fname)
                df_reviews.to_csv(fname, index=False, encoding="utf-8-sig")
                print(f"  ▶ CSV 저장 완료: {fname} (rows={len(df_reviews)})")
            else:
                print("  ▶ 파싱된 리뷰 없음 → 파일 미생성")

    finally:
        driver.quit()

## 네이버 블로그 본문 수집

---

In [ ]:
# -----------------------------------------------------
# 2) 100대 명산 키워드
# -----------------------------------------------------
keywords = [
    '강원도 가리산 등산후기', '강원도 가리왕산 등산후기', '경상남도 가야산 등산후기', '울산광역시 가지산 등산후기',
    '경기도 감악산 등산후기', '전라북도 강천산 등산후기', '대전광역시 계룡산 등산후기', '강원도 계방산 등산후기',
    '강원도 공작산 등산후기', '서울특별시 관악산 등산후기', '경상북도 구병산 등산후기', '경상남도 금산 등산후기',
    '충청북도 금수산 등산후기', '경상북도 금오산 등산후기', '부산광역시 금정산 등산후기', '전라남도 깃대봉 등산후기',
    '경상북도 남산(금오산) 등산후기', '경상북도 내연산 등산후기', '전라북도 내장산(신선봉) 등산후기', '충청남도 대둔산 등산후기',
    '강원도 대암산 등산후기', '경상북도 대야산 등산후기', '충청남도 덕숭산(수덕산) 등산후기', '전라북도 덕유산(향적봉) 등산후기',
    '강원도 덕항산 등산후기', '충청북도 도락산 등산후기', '서울특별시 도봉산(자운봉) 등산후기', '전라남도 두륜산 등산후기',
    '강원도 두타산 등산후기', '인천광역시 마니산 등산후기', '전라북도 마이산(암마이산) 등산후기', '강원도 명성산 등산후기',
    '경기도 명지산 등산후기', '전라북도 모악산 등산후기', '광주광역시 무등산 등산후기', '경상남도 무학산 등산후기',
    '경상남도 미륵산 등산후기', '충청북도 민주지산 등산후기', '전라북도 방장산 등산후기', '강원도 방태산(주억봉) 등산후기',
    '강원도 백덕산 등산후기', '전라북도 백암산 등산후기', '경기도 백운산(포천) 등산후기', '전라남도 백운산(광양) 등산후기',
    '강원도 백운산(정선) 등산후기', '전라북도 변산(의상봉) 등산후기', '서울특별시 북한산(백운대) 등산후기', '대구광역시 비슬산(천왕봉) 등산후기',
    '강원도 삼악산 등산후기', '충청남도 서대산 등산후기', '전라북도 선운산 등산후기', '강원도 설악산(대청봉) 등산후기',
    '경상북도 성인봉 등산후기', '경상북도 소백산 등산후기', '경기도 소요산 등산후기', '경상북도 속리산 등산후기',
    '울산광역시 신불산 등산후기', '경상남도 연화산 등산후기', '강원도 오대산(비로봉) 등산후기', '강원도 오봉산 등산후기',
    '경기도 용문산 등산후기', '강원도 용화산 등산후기', '경상북도 운문산 등산후기', '경기도 운악산(현등산) 등산후기',
    '전라북도 운장산 등산후기', '충청북도 월악산 등산후기', '전라남도 월출산 등산후기', '경기도 유명산 등산후기',
    '강원도 응봉산(매봉산) 등산후기', '전라북도 장안산 등산후기', '경상남도 재약산 등산후기', '전라북도 적상산 등산후기',
    '강원도 점봉산 등산후기', '전라남도 조계산 등산후기', '경상북도 주왕산 등산후기', '경상북도 주흘산 등산후기',
    '경상남도 지리산(천왕봉) 등산후기', '경상남도 통영시 지리산 등산후기', '전라남도 천관산 등산후기', '경기도 천마산 등산후기',
    '경상남도 천성산 등산후기', '충청북도 천태산 등산후기', '경상북도 청량산 등산후기', '전라남도 추월산 등산후기',
    '경기도 축령산 등산후기', '강원도 치악산 등산후기', '충청남도 칠갑산 등산후기', '강원도 태백산 등산후기',
    '강원도 태화산 등산후기', '경상북도 팔공산 등산후기', '강원도 팔봉산 등산후기', '전라남도 팔영산 등산후기',
    '제주도 한라산 등산후기', '경기도 화악산 등산후기', '경상남도 화왕산 등산후기', '경상남도 황매산 등산후기',
    '경상남도 황석산 등산후기', '경상북도 황악산 등산후기', '경상북도 황장산 등산후기', '경상북도 희양산 등산후기'
]

In [ ]:
# -----------------------------------------------------
# 4) 데이터 수집
# -----------------------------------------------------
print(f"100대 명산 블로그 크롤링 시작 ({len(keywords)}개 산)")
print("=" * 60)

all_results = []

for idx, keyword in enumerate(keywords, 1):
    print(f"[{idx}/{len(keywords)}] {keyword}")
    
    try:
        response = requests.get(
            "https://openapi.naver.com/v1/search/blog.json",
            params={'query': keyword, 'display': 100, 'start': 1, 'sort': 'sim'},
            headers={'X-Naver-Client-Id': CLIENT_ID, 'X-Naver-Client-Secret': CLIENT_SECRET},
            timeout=10
        )
        
        if response.status_code == 200:
            items = response.json().get('items', [])
            for item in items:
                all_results.append({
                    'keyword': keyword,
                    'mountain_name': keyword.split()[1].replace('등산후기', ''),
                    'region': keyword.split()[0],
                    'title': clean_html(item.get('title', '')),
                    'link': item.get('link', ''),
                    'description': clean_html(item.get('description', '')),
                    'bloggername': item.get('bloggername', ''),
                    'postdate': format_date(item.get('postdate', '')),
                })
            print(f"  ✓ {len(items)}건 수집")
        else:
            print(f"  ✗ API 오류 (status: {response.status_code})")
        
        time.sleep(0.1)
    except Exception as e:
        print(f"  ✗ 오류: {e}")

In [ ]:
# -----------------------------------------------------
# 5) 데이터 정리 및 저장
# -----------------------------------------------------
print(f"\n데이터 정리 중...")
df = pd.DataFrame(all_results)
df.drop_duplicates(subset=['link'], inplace=True)

print(f"총 수집: {len(df)}건 ({df['mountain_name'].nunique()}개 산)")
print(f"기간: {df['postdate'].min()} ~ {df['postdate'].max()}")

# CSV 저장
filename = f'mountains_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df.to_csv(filename, index=False, encoding='utf-8-sig')
print(f"\n✓ 저장 완료: {filename}")

# 본문 크롤링 (선택사항)
crawl_content = input("\n본문도 수집? (y/n): ").lower() == 'y'

if crawl_content:
    print(f"\n본문 크롤링 시작 ({len(df)}건)")
    df['full_content'] = None
    
    for idx, row in df.iterrows():
        if 'blog.naver.com' in row['link']:
            print(f"[{idx+1}/{len(df)}] {row['mountain_name']} 수집중...")
            content = extract_blog_content(row['link'])
            df.at[idx, 'full_content'] = content if content else "[실패]"
            time.sleep(1.5)
    
    # 재저장
    filename_full = 'all_mountains.csv'
    df.to_csv(filename_full, index=False, encoding='utf-8-sig')
    print(f"\n✓ 본문 포함 저장: {filename_full}")

print("\n완료!")

## 유튜브 댓글 수집

---

In [ ]:
# -----------------------------------------------------
# 산 이름 불러오기
# -----------------------------------------------------
with open("mountains.txt", "r", encoding="utf-8") as f:   # 산이름 + 지역명
    mountains = [line.strip() for line in f if line.strip()]

# -----------------------------------------------------
# 산별 댓글 + 자막 수집
# -----------------------------------------------------
for mountain in mountains:
    search_query = f"{mountain} 등산 후기"
    print(f"[INFO] {search_query} 검색 중...")

    # 상위 10개 영상 가져오기
    search_response = youtube.search().list(
        q=search_query,
        part="snippet",
        type="video",
        maxResults=10
    ).execute()

    items = search_response.get("items")
    if not items:
        print(f"[WARN] {search_query} 관련 영상 없음")
        continue

    all_comments = []
    all_captions = []

    for idx, video in enumerate(items, start=1):
        video_id = video["id"]["videoId"]
            
        try:
            comments = get_youtube_comments(video_id, max_comments=100)
            print(f"[INFO] {search_query} - 영상 {idx}: {len(comments)}개 댓글 수집 완료")
            all_comments.extend(comments)
        except Exception as e:
            print(f"[WARN] {search_query} - 영상 {idx}: 댓글 수집 불가 ({e})")
            
        try:
            transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['ko'])
            caption_text = " ".join([x['text'] for x in transcript])
            all_captions.append(caption_text)
            print(f"[INFO] {search_query} - 영상 {idx}: 자막 수집 완료")
        except Exception as e:
            print(f"[WARN] {search_query} - 영상 {idx}: 자막 없음 ({e})")
            
        time.sleep(1)  # API 제한 방지

    df = pd.DataFrame({
        "comments": all_comments + [None]*(len(all_captions)-len(all_comments)),  # 길이 맞추기
        "captions": all_captions + [None]*(len(all_comments)-len(all_captions))
    })
    filename = f"{mountain}_youtube.csv"
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"[INFO] {search_query} 모든 영상 데이터 CSV 저장 완료: {filename}")
    
    time.sleep(2)

## 카카오 맵 리뷰 크롤링

---

In [ ]:
if __name__ == "__main__":
    main_km()

## 네이버 지도 리뷰 크롤링

---

In [ ]:
# 실행
if __name__ == "__main__":
    main_nm()

## 합쳐서 마스터 테이블 구축

네이버 블로그 + 유튜브 댓글 + 카카오맵 리뷰 + 네이버지도 리뷰 

---

In [ ]:
# --------------------------------------------------------------------------------------
# 네이버 블로그 통합본
# --------------------------------------------------------------------------------------
blog_all = pd.read_csv("네이버_블로그/all_mountains.csv")
blog_all['mountain_name'].nunique()

In [ ]:
# --------------------------------------------------------------------------------------
# 유튜브 통합본 생성하기
# --------------------------------------------------------------------------------------

youtube_all = pd.concat(
    [
        pd.read_csv(f).assign(
            source_file=os.path.basename(f),   
            mountain=Path(f).stem              
        )
        for f in glob.glob(r"유투브_댓글/*.csv")
    ],
    ignore_index=True
)

In [ ]:
# --------------------------------------------------------------------------------------
# 카카오 지도 리뷰 통합본 생성하기
# --------------------------------------------------------------------------------------

kakaomap_all = pd.concat(
    [
        pd.read_csv(f).assign(
            source_file=os.path.basename(f),   
            mountain=Path(f).stem              
        )
        for f in glob.glob(r"네이버지도_카카오맵_지도리뷰/kakaomap_review/*.csv")
    ],
    ignore_index=True
)
# 파일 확장명이 km_가리산_review.csv 이런식
kakaomap_all['mountain'] = kakaomap_all['mountain'].str.split("_").str[1]

In [ ]:
# --------------------------------------------------------------------------------------
# 네이버 지도 리뷰 통합본 생성하기
# --------------------------------------------------------------------------------------

navermap_all = pd.concat(
    [
        pd.read_csv(f).assign(
            source_file=os.path.basename(f),   
            mountain=Path(f).stem              
        )
        for f in glob.glob(r"네이버지도_카카오맵_지도리뷰/navermap_review/*.csv")
    ],
    ignore_index=True
)
# 파일 확장명이 km_가리산_review.csv 이런식
navermap_all['mountain'] = navermap_all['mountain'].str.split("_").str[1]

In [ ]:
# --------------------------------------------------------------------------------------
# 100대 명산 이름 같은지 확인
# --------------------------------------------------------------------------------------

blog_set    = set(blog_all["mountain_name"])
youtube_set = set(youtube_all["mountain"])
kakao_set   = set(kakaomap_all["mountain"])
naver_set   = set(navermap_all["mountain"])


print("blog:", len(blog_set))
print("youtube:", len(youtube_set))
print("kakaomap:", len(kakao_set))
print("navermap:", len(naver_set))

# ----------------------------------------------------------------------------------------

# --- 변경 전 ---
blog_set    = set(blog_all["mountain_name"])
youtube_set = set(youtube_all["mountain"])
kakao_set   = set(kakaomap_all["mountain"])
naver_set   = set(navermap_all["mountain"])

print("=" * 20, "변경전", "=" * 20)
union_all  = blog_set | youtube_set | kakao_set | naver_set
common_all = blog_set & youtube_set & kakao_set & naver_set
print("UNION:", len(union_all), "COMMON:", len(common_all))
print(sorted(union_all - common_all))


# ----------------------------------------------------------------------------------------
# 네이버 블로그 크롤링 할 때 지리산 -> 통영시로 저장을 해서 다시 지리산으로 변경

# --- 값 변경 ---
blog_all["mountain_name"] = blog_all["mountain_name"].replace({"통영시": "지리산"})

blog_set = set(blog_all["mountain_name"].dropna())

print("=" * 20, "변경후", "=" * 20)
union_all  = blog_set | youtube_set | kakao_set | naver_set
common_all = blog_set & youtube_set & kakao_set & naver_set
print("UNION:", len(union_all), "COMMON:", len(common_all))
print(sorted(union_all - common_all))

In [ ]:
# --------------------------------------------------------------------------------------
# 자연어 분석에 필요한 컬럼 선정 및 생성
# --------------------------------------------------------------------------------------

# 블로그
blog_all_2 = blog_all[['mountain_name', 'full_content']]
blog_all_2["data_source"] = "naver_blog"

# 카카오 맵
kakaomap_all_2 = kakaomap_all[['mountain', 'km_content']]
kakaomap_all_2["data_source"] = "kakao_map"
kakaomap_all_2.rename(columns={'km_content' : 'full_content', 'mountain' : 'mountain_name'}, inplace= True)

# 네이버 맵
navermap_all_2 = navermap_all[['mountain', 'nm_content']]
navermap_all_2["data_source"] = "naver_map"
navermap_all_2.rename(columns={'nm_content' : 'full_content', 'mountain' : 'mountain_name'}, inplace= True)

# 유튜브 맵
youtube_all_2 = youtube_all[['mountain', 'comments']]
youtube_all_2["data_source"]  = "youtube"
youtube_all_2.rename(columns={'comments' : 'full_content', 'mountain' : 'mountain_name'}, inplace= True)

In [ ]:
# --------------------------------------------------------------------------------------
# 4개 데이터 결합
# --------------------------------------------------------------------------------------

df_list = [blog_all_2, kakaomap_all_2, navermap_all_2, youtube_all_2]
df = pd.concat(df_list, ignore_index=True)

# --------------------------------------------------------------------------------------
# review_id 생성
# --------------------------------------------------------------------------------------

df["review_id"] = (
    df["mountain_name"].astype(str) + "_" + df["data_source"].astype(str) + "_" +
    (df.groupby(["mountain_name","data_source"]).cumcount() + 1).astype(str)
)


In [ ]:
# 대암산 고도값 없음, 소요시간/난이도 계산 불가로 제외
df = df[df['mountain_name'] != "대암산"]

df.to_csv("99대명산_크롤링.csv", index = False)

In [ ]:
 # NLP 분석으로 넘어감